In [ ]:
# process the catalog to save M200c, R200c from Mvir using colossus
from astropy.io import fits
import colossus
from colossus.cosmology import cosmology as cosmology_colossus
from colossus.lss import mass_function
from colossus.halo import concentration
from colossus.halo import mass_defs
from scipy.ndimage.filters import gaussian_filter1d
params = {'w0':-1.0 ,'flat': True, 'H0': 70.0, 'Om0': 0.286, 'Ob0': 0.047, 'sigma8':0.82 ,'ns': 0.96}
cosmology_colossus.addCosmology('myCosmo', params)
cosmology_colossus.setCosmology('myCosmo')
cosmo_colossus = cosmology_colossus.setCosmology('myCosmo')

hcat_fname = '/mnt/home/spandey/ceph/GODMAX/data/cardinal/halo_input.fits'
catalog_data = fits.open(hcat_fname)[1].data
zh_all = catalog_data['z']
mvir_all = np.array(catalog_data['m200'], dtype=np.float64)


z0 = 0.0
zf = 0.95
nz = 100
zedges = np.linspace(z0, zf, nz+1)
M200c_all = np.zeros_like(mvir_all)
c200c_all = np.zeros_like(mvir_all)
for jz in range(nz):
    z0_jz = zedges[jz]
    zf_jz = zedges[jz+1]
    indsel = np.where((zh_all >= z0_jz) & (zh_all < zf_jz))[0]
    mvir_sel = mvir_all[indsel]
    zsel = zh_all[indsel]
    zbar = np.mean(zsel)
    cvir_sel = concentration.concentration(mvir_sel, 'vir', zbar, model = 'ishiyama21')
    M200c_sel, R200c_sel, c200c_sel = mass_defs.changeMassDefinition(mvir_sel, cvir_sel, zbar, 'vir', '200c')
    M200c_all[indsel] = M200c_sel
    c200c_all[indsel] = c200c_sel


